# Laya V3.8 — Step-by-Step Layer Optimizer: Faster AND More Accurate

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V38_Greedy_Layerwise_Fast_Accurate_Colab.ipynb)

**Objective:** inspect **all 28 pretrained ModernBERT layers**, propose a cheaper form of every layer, **accept changes cumulatively only if they preserve or improve decisions and measured CUDA speed**, and roll back failures. This is a **zero-shot structural ablation / greedy architecture search**, **not** a guarantee that a smaller or more accurate model exists without retraining.

### What V3.7 established
- Original attention has **10 full-attention + 18 sliding-attention layers**; the true pretrained sliding radius is **64 tokens**. The wrong sliding-mask interface in V3.6 was corrected and its exact reconstruction passed.
- V3.7 probe: replacing *every* sliding radius 64 with radius 32 achieved **93.3%** teacher agreement (60 decisions) and **3.3 percentage points** gold-accuracy loss; deleting all attention catastrophically reduced accuracy. Those configurations fail the gate. Individual layers, however, **were not scanned**.
- Native T4 timing varies considerably across repeated measurements. A candidate must be benchmarked against its predecessor in an **interleaved, matched-batch** protocol, not compared against stale timings.

### Exact execution strategy

1. **One frozen pretrained model** on GPU, no second copy; cache original teacher probability vectors on CPU. Maintain a reversible map of layer replacements. No \`QKV\`, RoPE, MLP or attention weight cloning.
2. **Safety preflight:** full and sliding native replay; reconstruct native sliding mask with original radius 64. Verify candidate changes really replace exactly the intended layer, and rollback leaves native predictions unchanged.
3. **Fast local scan** (all 28, light probe): for each original sliding layer, try \`band_r48\`, \`band_r32\`, \`skip_attention\` and \`skip_block\`. For each full layer, try \`band_r384\`, \`band_r256\`, \`skip_attention\` and \`skip_block\`. Probe score preserves original decisions first.
4. **Cumulative greedy pass:** visit all 28 layers in order, evaluate their strongest pre-screened options *against the current accepted model*. If gold accuracy falls below the original or teacher agreement <95%, **reject**. Otherwise benchmark candidate vs current architecture on the exact same examples; require **minimum 2% paired speedup**. Also allow measured accuracy gains with no meaningful speed regression if configured, but keep the speed-first default.
5. **Reverse refinement:** recheck accepted layer options from last to first; reversible conditional modifications guard against layer interactions.
6. **Untouched audit:** only the final configuration and the original reference receive full held-out audit. Test agreement, gold accuracy, JS and paired inference latency. If it fails gates, **revert deployment recommendation to native**.
7. Save **ablation matrix, accepted/rejected decision log, selected per-layer blueprint**, inference metrics and estimated parameter savings for actually skipped layers. Optional independent accuracy-enhancement training is **not performed**.

### Crucial distinction

A shorter masked attention band does not necessarily reduce CUDA compute with native SDPA, because custom masks may still run a dense kernel. Actual speed savings are most likely from removing attention or whole residual blocks that do little for these tasks. No modified model is claimed faster until it passes paired GPU timing *and* held-out accuracy. The original checkpoint and official TEST split remain untouched.

**Default budget**: 28 layers × lightweight candidate pre-screen on 36 PROBE decisions, greedy acceptance on 120 PROBE decisions, 200 AUDIT decisions, only 3 GPU timing batches. All four alternatives per layer are evaluated by default; reduce MAX_PROMOTED_PER_LAYER to 2 for a quicker exploratory run. Increase probe/audit sizes for publication quality. Colab GPU runtime required.

Notebook generated from the verified V3.7 data pipeline and native ModernBERT 4.57.6 attention interfaces.

### Improvement: Structural-first resumable search
This revision prioritizes `skip_block` and `zero` candidates when the short PROBE screen indicates good fidelity, rather than spending the early optimization phase on smaller masked windows that remain **dense SDPA**. Every completed screening and greedy layer is persisted to `v38_resume_checkpoint.json`, and rerunning the notebook resumes completed work. By default, progress lives under `/content` (survives cell reruns but **not runtime deletion**). Enable `PERSIST_TO_GOOGLE_DRIVE` before the first run to persist across runtime resets. The reverse pass is optional and disabled for faster initial results.

**Do not** interpret 36-decision isolated agreement as certification to prune: accepted changes must pass cumulative 120-decision PROBE checks and independent 200-decision AUDIT; timing is paired and interleaved, and a 2% speed ratio is treated as exploratory, not statistically definitive.


In [ ]:
#@title 1. Install dependencies and GPU preflight
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(),"Select Runtime > Change runtime type > GPU"
print("PyTorch",torch.__version__,"CUDA",torch.version.cuda,
      "GPU",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2038
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 80 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 1 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 2 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 2 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V38_GreedyLayerwise")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4 and TRAIN_CACHE_BATCHES > 0
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
# student_eval not built: use a single model and temporarily swap wrappers
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
# single teacher model only
teacher.encoder.config.reference_compile = False
# one model; reference_compile disabled above

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
#@title V3.8 performance-quality search controls
SCREEN_DECISIONS=36 #@param {type:"integer"}
SEARCH_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
MAX_PROMOTED_PER_LAYER=4 #@param {type:"integer"}
PRIORITIZE_STRUCTURAL=True #@param {type:"boolean"}
RESUME_SEARCH=True #@param {type:"boolean"}
RESET_SAVED_SEARCH=False #@param {type:"boolean"}
PERSIST_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}
RUN_REVERSE_REFINEMENT=False #@param {type:"boolean"}
INTERLEAVED_REPEATS=4 #@param {type:"integer"}
MIN_PAIRED_SPEEDUP=1.02 #@param {type:"number"}
ALLOW_ACCURACY_FIRST=False #@param {type:"boolean"}
MAX_SEARCH_GOLD_DROP_PP=0.0 #@param {type:"number"}
MAX_FINAL_GOLD_DROP_PP=1.0 #@param {type:"number"}
MIN_TEACHER_AGREEMENT=.95 #@param {type:"number"}
EXPORT_ACCEPTED_BLUEPRINT=True #@param {type:"boolean"}
if PERSIST_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR=Path("/content/drive/MyDrive/TinyCeNN-LM/v38_search")
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CHECKPOINT_PATH=OUTPUT_DIR/"v38_resume_checkpoint.json"
assert 12<=SCREEN_DECISIONS<=SEARCH_DECISIONS<=400
assert SEARCH_DECISIONS<=len(probe_items) if "probe_items" in globals() else True
assert 20<=AUDIT_DECISIONS<=400
assert INTERLEAVED_REPEATS>=2 and MIN_PAIRED_SPEEDUP>=1.
assert MAX_PROMOTED_PER_LAYER>=1
ALL_LAYERS=list(range(len(kinds)))
FULL_LAYERS=[i for i,k in enumerate(kinds) if k=="full_attention"]
SLIDING_LAYERS=[i for i in ALL_LAYERS if i not in FULL_LAYERS]
assert len(ALL_LAYERS)==28 and len(FULL_LAYERS)==10 and len(SLIDING_LAYERS)==18
ORIGINAL_LOCAL_RADIUS=int(teacher.encoder.config.local_attention)//2
print("Original sliding radius",ORIGINAL_LOCAL_RADIUS,"full layers",len(FULL_LAYERS),
      "sliding layers",len(SLIDING_LAYERS),"model GPU copies: 1")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.8 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Mask-only and block-skip patches, with cumulative reversible installation
from functools import lru_cache
from contextlib import contextmanager

# Original ModernBERT 4.57.6 SDPA consumes separate masks:
# full attention -> attention_mask; sliding attention -> sliding_window_mask.
# Preserve the native module itself: no QKV/Wo/RoPE cloning or alternative SDPA.
# The original encoder creates both 4D additive masks before visiting layers.

@lru_cache(maxsize=72)
def cached_band(length,radius,dev):
    ar=torch.arange(length,device=torch.device(dev))
    return ((ar[:,None]-ar[None,:]).abs()<=radius)[None,None,:,:]

def narrowed_mask(original_global,radius,T):
    if original_global is None:
        raise RuntimeError("Expected original encoder's 4D additive global mask.")
    if original_global.ndim!=4 or original_global.shape[-2:]!=(T,T):
        raise RuntimeError(f"Unexpected native global mask {original_global.shape}")
    band=cached_band(T,int(radius),str(original_global.device))
    if original_global.dtype==torch.bool:
        return original_global & band
    return original_global.masked_fill(~band,torch.finfo(original_global.dtype).min)

class AttentionMaskAblation(nn.Module):
    """Zero-copy (parameters owned by original module) native-SDPA mask shim."""
    def __init__(self,original,original_kind,mode="band",radius=128):
        super().__init__()
        self.original=original
        self.kind=original_kind
        self.mode=mode
        self.radius=int(radius)

    def forward(self,hidden_states,*args,**kwargs):
        if self.mode=="zero":
            # Skip only attention residual: encoder FFN, norms, classifier stay.
            return (torch.zeros_like(hidden_states),)
        if self.mode=="passthrough":
            return self.original(hidden_states,*args,**kwargs)
        original_global=kwargs.get("attention_mask")
        original_slide=kwargs.get("sliding_window_mask")
        if self.mode=="band":
            changed=narrowed_mask(original_global,self.radius,hidden_states.shape[1])
            if self.kind=="full_attention":
                kwargs=dict(kwargs,attention_mask=changed)
            else:
                kwargs=dict(kwargs,sliding_window_mask=changed)
        elif self.mode=="global":
            if self.kind=="full_attention":
                return self.original(hidden_states,*args,**kwargs)
            kwargs=dict(kwargs,sliding_window_mask=original_global)
        else:
            raise ValueError(self.mode)
        return self.original(hidden_states,*args,**kwargs)

original_attention=[layer.attn for layer in teacher.encoder.layers]


# full block skip is a DIFFERENT ablation from attention-only skip.
# It eliminates the attention residual, layer norms and the MLP for
# the selected layer, while retaining the original residual stream.
class SkipResidualBlock(nn.Module):
    def forward(self,hidden_states,*args,**kwargs):
        return (hidden_states,)

original_layers=tuple(teacher.encoder.layers)
original_attention=tuple(layer.attn for layer in original_layers)

@contextmanager
def active_plan(plan):
    """Temporarily install *cumulative* layer choices, then restore EXACT originals.

    plan maps layer indices to ("band",radius), ("zero",0) or ("skip_block",0).
    A complete plan has an entry only for layers changed from native.
    """
    assert all(teacher.encoder.layers[i] is original_layers[i]
               for i in ALL_LAYERS),"Native modules were not restored."
    try:
        for i in sorted(plan):
            mode,radius=plan[i]
            if mode=="skip_block":
                teacher.encoder.layers[i]=SkipResidualBlock()
            else:
                assert mode in ("band","zero")
                original_layers[i].attn=AttentionMaskAblation(
                    original_attention[i],kinds[i],mode=mode,radius=radius
                )
        teacher.eval()
        yield teacher
    finally:
        for i in ALL_LAYERS:
            original_layers[i].attn=original_attention[i]
            teacher.encoder.layers[i]=original_layers[i]
        teacher.eval()
        assert all(teacher.encoder.layers[i] is original_layers[i] and
                   original_layers[i].attn is original_attention[i]
                   for i in ALL_LAYERS),"Rollback failed"

def plan_description(plan):
    return {str(i):dict(mode=mode,radius=radius,original_kind=kinds[i])
            for i,(mode,radius) in sorted(plan.items())}
def variants_for_layer(i):
    if kinds[i]=="full_attention":
        return (("band",384),("band",256),("zero",0),("skip_block",0))
    return (("band",48),("band",32),("zero",0),("skip_block",0))
print("One-copy cumulative patching initialized for all",len(ALL_LAYERS),"layers.")


In [ ]:
#@title 5. Mandatory correct native full/sliding mask replay before optimization
# Run before any experiment: a bad native mask would invalidate every result.
# Source ModernBERT uses distance <= config.local_attention // 2,
# NOT a centered 256-token total-width convention.

b=batch(probe_items,0,BATCH)
for i in (FULL_LAYERS[0],SLIDING_LAYERS[0],SLIDING_LAYERS[-1]):
    rec=attn_io(teacher,i,b)
    original_global=rec["call_kwargs"]["attention_mask"]
    original_sliding=rec["call_kwargs"]["sliding_window_mask"]
    assert original_global.ndim==4 and original_sliding.ndim==4
    T=rec["x"].shape[1]
    rebuilt=narrowed_mask(original_global,ORIGINAL_LOCAL_RADIUS,T)
    # The EXACT original mask must be reconstructible from the global mask.
    assert torch.equal(rebuilt,original_sliding),(
        "Original sliding mask mismatch — do not proceed",i,
        float((rebuilt.float()-original_sliding.float()).abs().max())
    )
    for mode,radius in (("passthrough",ORIGINAL_LOCAL_RADIUS),
                        ("band",ORIGINAL_LOCAL_RADIUS)):
        # A full-attention layer is not expected to be identical after
        # changing its mask. Native parity on full: passthrough only.
        if kinds[i]=="full_attention" and mode=="band":continue
        wrapper=AttentionMaskAblation(
            original_attention[i],kinds[i],mode=mode,radius=radius
        )
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
            replay=_attention_output_tensor(wrapper(
                *rec["call_args"],**rec["call_kwargs"]
            ))
        diff=float((replay.float()-rec["y"].float()).abs().max())
        assert diff<1e-2,(i,mode,diff)
        print("Native SDPA masking parity PASS",kinds[i],
              "layer",i,mode,"maxΔ",diff)
print("Mask path + exact original sliding radius verified.")

# The patched graph must roll back to the exact object identity.
with active_plan({SLIDING_LAYERS[0]:("band",ORIGINAL_LOCAL_RADIUS)}):
    assert teacher.encoder.layers[SLIDING_LAYERS[0]].attn is not original_attention[SLIDING_LAYERS[0]]
assert teacher.encoder.layers[SLIDING_LAYERS[0]].attn is original_attention[SLIDING_LAYERS[0]]
with active_plan({SLIDING_LAYERS[0]:("skip_block",0)}):
    assert isinstance(teacher.encoder.layers[SLIDING_LAYERS[0]],SkipResidualBlock)
assert teacher.encoder.layers[SLIDING_LAYERS[0]] is original_layers[SLIDING_LAYERS[0]]
print("Cumulative patch rollback and structural skip preflight PASS.")


In [ ]:
#@title 6. One-copy baseline prediction cache for fast screening and untouched audit
@torch.inference_mode()
def predictions(items,limit):
    out=[]
    for first in range(0,min(limit,len(items)),BATCH):
        segment=items[first:min(first+BATCH,limit)]
        batch_inputs=direct_batch(segment)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            scores,_=teacher(**batch_inputs)
        allowed=batch_inputs["marker_mask"].bool()
        p=torch.softmax(scores.float().masked_fill(~allowed,-1e9),dim=-1)
        for j,item in enumerate(segment):
            out.append(dict(
                probabilities=p[j,:].cpu(),
                decision=int(p[j].argmax()),
                gold=int(item["gold_index"]),
                case_id=item["case_id"],qid=item["qid"],
                token_length=len(item["ids"])
            ))
    return out

probe_ref=predictions(probe_items,SEARCH_DECISIONS)
audit_ref=predictions(audit_items,AUDIT_DECISIONS)
assert len(probe_ref)==SEARCH_DECISIONS and len(audit_ref)==AUDIT_DECISIONS
assert {x["case_id"] for x in probe_ref}.isdisjoint(
    {x["case_id"] for x in audit_ref}
)
print("Cached",len(probe_ref),"probe and",len(audit_ref),
      "audit native predictions on CPU; no other model allocated.")
print("Original teacher gold accuracy, audit:",
      sum(r["decision"]==r["gold"] for r in audit_ref)/len(audit_ref))

screen_ref=probe_ref[:SCREEN_DECISIONS]
screen_items=probe_items[:SCREEN_DECISIONS]
search_items=probe_items[:SEARCH_DECISIONS]
print("Screen",len(screen_ref),"greedy search",len(probe_ref),
      "untouched audit",len(audit_ref),"disjoint cases across probe and audit.")


In [ ]:
#@title 7. Efficient GPU decision metrics and fully PAIRED interleaved inference latency
@torch.inference_mode()
def evaluate_plan(plan,items,reference):
    """Candidate-versus-original teacher. No second checkpoint in GPU memory."""
    n=min(len(items),len(reference))
    agree=correct=gold_teacher=0
    total_js=0.
    changes=[]
    with active_plan(plan):
        for first in range(0,n,BATCH):
            chunk=items[first:min(n,first+BATCH)]
            data=direct_batch(chunk)
            with torch.autocast("cuda",dtype=amp_dtype):
                logits,_=teacher(**data)
            visible=data["marker_mask"].bool()
            out=torch.softmax(logits.float().masked_fill(~visible,-1e9),dim=-1).cpu()
            for off,item in enumerate(chunk):
                idx=first+off
                base=reference[idx]
                p=base["probabilities"]; q=out[off]
                assert len(p)==len(q)
                mean=(p+q)*.5
                total_js+=float(.5*((p*(p.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum()+
                                   (q*(q.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum()))
                prediction=int(q.argmax())
                agree+=int(prediction==base["decision"])
                correct+=int(prediction==item["gold_index"])
                gold_teacher+=int(base["decision"]==item["gold_index"])
                if prediction!=base["decision"]:
                    changes.append(dict(case_id=item["case_id"],qid=item["qid"],
                                        gold=int(item["gold_index"]),
                                        native=base["decision"],optimized=prediction,
                                        seq_len=len(item["ids"])))
    return dict(
        decisions=n,teacher_agreement=agree/max(n,1),
        teacher_gold_accuracy=gold_teacher/max(n,1),
        student_gold_accuracy=correct/max(n,1),
        gold_accuracy_drop=(gold_teacher-correct)/max(n,1),
        mean_js=total_js/max(n,1),
        different_predictions=changes[:25],
    )

# Pre-collate identical token shapes for candidates.
timing_batches=[batch(probe_items,i*BATCH,BATCH) for i in (0,7,13)]
timing_shapes=[(int(b["input_ids"].shape[0]),int(b["input_ids"].shape[1]))
               for b in timing_batches]

@torch.inference_mode()
def benchmark_paired(plan_a,plan_b,repeats=INTERLEAVED_REPEATS):
    """Interleave A/B/B/A in the same GPU session and compare matched token shapes.

    Kernel + Python launch overhead are captured by CUDA events on an identical
    input batch. Each repetition reuses the same tensors for both variants.
    Excludes checkpoint loading/compilation. Returns median paired ratio.
    """
    rows=[]
    for b in timing_batches:
        def run(plan,iterations=1):
            with active_plan(plan):
                with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
                    for _ in range(iterations):
                        teacher(**b)
                    torch.cuda.synchronize()
                    start=torch.cuda.Event(enable_timing=True)
                    end=torch.cuda.Event(enable_timing=True)
                    start.record()
                    teacher(**b)
                    end.record()
                    end.synchronize()
                    return float(start.elapsed_time(end))
        # warm both variants once; this also populates the backend kernel caches.
        run(plan_a,iterations=2);run(plan_b,iterations=2)
        for r in range(repeats):
            order=("a","b","b","a") if r%2==0 else ("b","a","a","b")
            record={"a":[],"b":[]}
            for label in order:
                record[label].append(run(plan_a if label=="a" else plan_b))
            ma=float(np.mean(record["a"]))
            mb=float(np.mean(record["b"]))
            rows.append(dict(batch=int(b["input_ids"].shape[0]),
                             tokens=int(b["input_ids"].shape[1]),
                             current_ms=ma,candidate_ms=mb,
                             speedup=ma/max(mb,1e-8)))
    ratios=[x["speedup"] for x in rows]
    return dict(speedup_median=float(np.median(ratios)),
                current_ms=float(np.median([x["current_ms"] for x in rows])),
                candidate_ms=float(np.median([x["candidate_ms"] for x in rows])),
                per_pair=rows)
# Strict no-op test (same exact model, no difference expected)
base=evaluate_plan({},search_items,probe_ref)
assert base["teacher_agreement"]==1 and base["mean_js"]<1e-8
print("Exact baseline teacher decisions PASS",base)
print("Matched CUDA timing batch/length:",timing_shapes)


In [ ]:
#@title 8. Durable checkpoint / resume for expensive all-layer optimization
import hashlib

# A progress file belongs to one exact checkpoint, data split and search setup.
signature=dict(
    source_model=SOURCE_MODEL,seed=SEED,
    screen_decisions=SCREEN_DECISIONS,search_decisions=SEARCH_DECISIONS,
    probe_ids=[(item["case_id"],item["qid"]) for item in search_items],
    gamma=(MAX_SEARCH_GOLD_DROP_PP,MIN_TEACHER_AGREEMENT,MIN_PAIRED_SPEEDUP),
    options=(MAX_PROMOTED_PER_LAYER,PRIORITIZE_STRUCTURAL,
             ALLOW_ACCURACY_FIRST,INTERLEAVED_REPEATS),
)
fingerprint=hashlib.sha256(
    json.dumps(signature,sort_keys=True).encode()
).hexdigest()

def save_progress():
    # Atomic write: partial Colab cell crashes must not corrupt the file.
    progress["fingerprint"]=fingerprint
    tmp=CHECKPOINT_PATH.with_suffix(".part")
    tmp.write_text(json.dumps(progress,indent=2,default=str),encoding="utf-8")
    tmp.replace(CHECKPOINT_PATH)

if RESET_SAVED_SEARCH and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
if RESUME_SEARCH and CHECKPOINT_PATH.exists():
    progress=json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    if progress.get("fingerprint")!=fingerprint:
        raise RuntimeError(
            "Existing V3.8 progress belongs to a different configuration or "
            "probe split. Set RESET_SAVED_SEARCH=True for a fresh run, or "
            "restore the original settings. No checkpoint data were reused."
        )
    print("RESUME enabled: found previous completed screens:",
          len(progress.get("screen_results",{})),
          "completed greedy layers:",
          len(progress.get("completed_greedy_layers",[])),
          "reverse:",
          len(progress.get("reverse_completed",[])))
else:
    progress=dict(fingerprint=fingerprint,screen_results={},
                  accepted_plan={},greedy_history=[],completed_greedy_layers=[],
                  reverse_history=[],reverse_completed=[],search_order=[])
    save_progress()
    print("New V3.8 search state created.")

def unpack_plan(obj):
    return {int(k):(v[0],int(v[1])) for k,v in obj.items()}

def sorted_layer_options(trials):
    # Same four tests, but place architecturally meaningful savings first
    # once a quick 36-decision screen says that they are plausible.
    preference={"skip_block":0,"zero":1,"band":2}
    def key(v):
        q=v["quality"]
        plausible=(q["teacher_agreement"]>=.95 and
                   q["gold_accuracy_drop"]<=.03)
        return (
            0 if plausible else 1,
            preference[v["mode"]] if plausible and PRIORITIZE_STRUCTURAL else 0,
            -q["teacher_agreement"],
            q["gold_accuracy_drop"],
            q["mean_js"],
            preference[v["mode"]],
        )
    return sorted(trials,key=key)

def search_priority(idx):
    # Prefer whole-block and attention-only removals that survived the
    # isolated screen; these can genuinely reduce GEMM launches and weights.
    choices=progress["screen_results"][str(idx)]
    best={}
    for row in choices:
        best[row["mode"]]=row["quality"]
    for mode in ("skip_block","zero"):
        q=best.get(mode)
        if q is not None and q["teacher_agreement"]>=.97 and q["gold_accuracy_drop"]<=.01:
            return (0 if mode=="skip_block" else 1,-q["teacher_agreement"],idx)
    return (2,0,idx)

def robust_speed(timing):
    ratios=timing["per_pair"]
    groups={}
    for row in ratios:
        groups.setdefault(row["tokens"],[]).append(row["speedup"])
    win_count=sum(np.median(scores)>=1.0 for scores in groups.values())
    needed=(len(groups)+1)//2
    return bool(timing["speedup_median"]>=MIN_PAIRED_SPEEDUP and
                win_count>=needed)

print("Progress file:",CHECKPOINT_PATH)
print("Cloud VM files survive cell reruns but not runtime deletion. "
      "Turn PERSIST_TO_GOOGLE_DRIVE on to survive new runtimes.")


In [ ]:
#@title 9. Independent 28-layer screen with checkpointing (resumes by layer)
screen_results=progress["screen_results"]
for idx in ALL_LAYERS:
    if str(idx) in screen_results:
        print(f"Screen L{idx:02d} restored from checkpoint; skipping.")
        continue
    trials=[]
    for mode,radius in variants_for_layer(idx):
        metrics=evaluate_plan({idx:(mode,radius)},screen_items,screen_ref)
        row=dict(layer=idx,kind=kinds[idx],mode=mode,radius=radius,
                 quality=metrics)
        trials.append(row)
        print(f"L{idx:02d} {kinds[idx]:<17} {mode:<10} r={radius:<3} "
              f"agree={metrics['teacher_agreement']:.3f} "
              f"gold_drop={metrics['gold_accuracy_drop']:+.3f} "
              f"JS={metrics['mean_js']:.5f}")
    screen_results[str(idx)]=sorted_layer_options(trials)
    save_progress()   # resilient even if Colab disconnects after layer 12
assert len(screen_results)==len(ALL_LAYERS)
screen_scores={str(i):sorted_layer_options(screen_results[str(i)])[
    :MAX_PROMOTED_PER_LAYER] for i in ALL_LAYERS}
order=(sorted(ALL_LAYERS,key=search_priority)
       if PRIORITIZE_STRUCTURAL else ALL_LAYERS.copy())
saved_order=progress.get("search_order") or []
if saved_order and saved_order!=order:
    raise RuntimeError("Saved search ordering does not match new screen.")
progress["search_order"]=order
save_progress()
print("\nScreen completed: 112 independent combinations.")
print("PRIORITY ORDER (most likely to speed up first):",order)
print("Potential structurally removable layers:",
      [i for i in order if search_priority(i)[0]<=1])


In [ ]:
#@title 10. Resumable GREEDY selection — prioritize real block removals
baseline_gold=base["student_gold_accuracy"]
accepted_plan=unpack_plan(progress.get("accepted_plan",{}))
greedy_history=progress.get("greedy_history",[])
completed=set(progress.get("completed_greedy_layers",[]))
search_order=progress["search_order"]
assert search_order and len(search_order)==len(ALL_LAYERS)
current_metrics=evaluate_plan(accepted_plan,search_items,probe_ref)
assert current_metrics["teacher_agreement"]>=MIN_TEACHER_AGREEMENT
assert current_metrics["student_gold_accuracy"]+MAX_SEARCH_GOLD_DROP_PP/100>=baseline_gold
print("Resume: accepted",plan_description(accepted_plan),
      "completed",len(completed),"/28.")
for idx in search_order:
    if idx in completed:
        print(f"L{idx:02d} already evaluated — restored; skipping.")
        continue
    choices=screen_scores[str(idx)]
    print(f"\n== Optimize L{idx:02d} ({kinds[idx]}) "
          f"with {len(accepted_plan)} previously accepted changes ==")
    succeeded=False
    for opt in choices:
        change=(opt["mode"],int(opt["radius"]))
        trial={**accepted_plan,idx:change}
        metrics=evaluate_plan(trial,search_items,probe_ref)
        quality_pass=(
            metrics["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            metrics["student_gold_accuracy"]+MAX_SEARCH_GOLD_DROP_PP/100>=baseline_gold and
            metrics["student_gold_accuracy"]+MAX_SEARCH_GOLD_DROP_PP/100>=
                current_metrics["student_gold_accuracy"]
        )
        result=dict(layer=idx,original_kind=kinds[idx],choice=change,
                    quality=metrics,accurate=bool(quality_pass),accepted=False)
        if not quality_pass:
            result["reason"]="insufficient teacher agreement or gold accuracy"
            greedy_history.append(result)
            print("  REJECT accuracy:",change,
                  "agree",round(metrics["teacher_agreement"],4),
                  "gold",round(metrics["student_gold_accuracy"],4))
            continue
        timing=benchmark_paired(accepted_plan,trial)
        reliable=robust_speed(timing)
        result["paired_timing"]=timing
        result["robust_speed_pass"]=reliable
        gold_gain=metrics["student_gold_accuracy"]>current_metrics["student_gold_accuracy"]
        allow_quality=ALLOW_ACCURACY_FIRST and gold_gain and timing["speedup_median"]>=.98
        if reliable or allow_quality:
            result["accepted"]=True
            result["reason"]="paired CUDA improvement" if reliable else "accuracy first"
            accepted_plan=trial
            current_metrics=metrics
            succeeded=True
            print("  ACCEPT",change,
                  "speedup",round(timing["speedup_median"],3),
                  "gold",round(metrics["student_gold_accuracy"],4))
            greedy_history.append(result)
            break
        result["reason"]="paired GPU speedup below threshold"
        greedy_history.append(result)
        print("  REJECT speed:",change,"ratio",round(timing["speedup_median"],3))
    if not succeeded:
        print("  KEEP ORIGINAL at",idx)
    completed.add(idx)
    progress["accepted_plan"]={str(i):list(v) for i,v in accepted_plan.items()}
    progress["greedy_history"]=greedy_history
    progress["completed_greedy_layers"]=sorted(completed)
    save_progress() # every completed layer, even unchanged
print("\nGreedy pass COMPLETE: accepted",len(accepted_plan),
      "structural/mask changes out of 28.")
print("Current plan:",plan_description(accepted_plan))


In [ ]:
#@title 11. Optional reverse refinement, checkpointed and interruption-safe
accepted_plan=unpack_plan(progress.get("accepted_plan",{}))
reverse_history=progress.get("reverse_history",[])
reverse_completed=set(progress.get("reverse_completed",[]))
if len(progress.get("completed_greedy_layers",[]))!=28:
    raise RuntimeError("Finish the forward greedy pass before refinement.")
if RUN_REVERSE_REFINEMENT:
    for idx in reversed(progress["search_order"]):
        if idx in reverse_completed:
            print("Refinement restored completed L",idx)
            continue
        prior=evaluate_plan(accepted_plan,search_items,probe_ref)
        for opt in screen_scores[str(idx)]:
            proposed=(opt["mode"],int(opt["radius"]))
            if accepted_plan.get(idx)==proposed:continue
            candidate={**accepted_plan,idx:proposed}
            q=evaluate_plan(candidate,search_items,probe_ref)
            qualified=(
                q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
                q["student_gold_accuracy"]+MAX_SEARCH_GOLD_DROP_PP/100>=baseline_gold and
                q["student_gold_accuracy"]+MAX_SEARCH_GOLD_DROP_PP/100>=
                    prior["student_gold_accuracy"]
            )
            if not qualified:continue
            timings=benchmark_paired(accepted_plan,candidate)
            accuracy_gain=(q["student_gold_accuracy"]>prior["student_gold_accuracy"])
            if robust_speed(timings) or (
                ALLOW_ACCURACY_FIRST and accuracy_gain and
                timings["speedup_median"]>=.98
            ):
                reverse_history.append(dict(layer=idx,
                    old=accepted_plan.get(idx),new=proposed,
                    paired_timing=timings,quality=q))
                accepted_plan=candidate
                print("Reverse ACCEPT L",idx,proposed,
                      "paired speedup",round(timings["speedup_median"],3))
                break
        reverse_completed.add(idx)
        progress["accepted_plan"]={str(i):list(v) for i,v in accepted_plan.items()}
        progress["reverse_history"]=reverse_history
        progress["reverse_completed"]=sorted(reverse_completed)
        save_progress()
    print("Reverse refinement COMPLETE:",len(reverse_history),"accepted revisions.")
else:
    print("Reverse refinement disabled (can enable and rerun later).")
print("PROBE final plan:",plan_description(accepted_plan))


In [ ]:
#@title 12. Strict one-time held-out audit + fair final GPU comparison
accepted_plan=unpack_plan(progress.get("accepted_plan",{}))
assert len(progress.get("completed_greedy_layers",[]))==28,(
    "Greedy forward pass incomplete. Resume from cell 10."
)
if RUN_REVERSE_REFINEMENT:
    assert len(progress.get("reverse_completed",[]))==28
if "final_audit_results" in progress:
    audit_record=progress["final_audit_results"]
    assert audit_record["selected_blueprint"]==plan_description(accepted_plan),(
        "The selected blueprint has changed. Clear only the cached final "
        "audit record and rerun this cell; don't choose based on audit."
    )
    final_probe=audit_record["probe"]
    final_audit=audit_record["audit"]
    final_timing=audit_record["paired_timing"]
    gates=audit_record["gates"]
    recommended_plan=accepted_plan if gates["all_pass"] else {}
    print("Restored final audit for unchanged blueprint; no repeat audit.")
else:
    final_probe=evaluate_plan(accepted_plan,search_items,probe_ref)
    final_audit=evaluate_plan(accepted_plan,audit_items[:AUDIT_DECISIONS],audit_ref)
    final_timing=benchmark_paired({},accepted_plan,
                                  repeats=max(6,INTERLEAVED_REPEATS))
    # The baseline compared with itself must NOT count as a speedup.
    speed_accept=bool(accepted_plan) and robust_speed(final_timing)
    gates=dict(
        agreement_ge_95=final_audit["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
        accuracy_drop_le_limit=
            final_audit["gold_accuracy_drop"]<=MAX_FINAL_GOLD_DROP_PP/100,
        reliable_paired_speedup=speed_accept,
    )
    gates["all_pass"]=all(gates.values())
    recommended_plan=accepted_plan if gates["all_pass"] else {}
    progress["final_audit_results"]=dict(
        selected_blueprint=plan_description(accepted_plan),
        probe=final_probe,audit=final_audit,
        paired_timing=final_timing,gates=gates)
    save_progress()
print("PROBE:",final_probe)
print("AUDIT:",final_audit)
print("Final matched CUDA:",final_timing)
print("Final acceptance gates:",gates)
if not gates["all_pass"]:
    print("No validated faster accurate replacement: retain native model.")
else:
    print("Provisional success; test official TEST, more seeds, and physical pruning.")


In [ ]:
#@title 13. Structural savings estimate and manifest export
# Masks only change attention behavior; native SDPA may remain dense.
# A skipped attention module saves runtime QKV/RoPE/Wo, but its original
# weights stay resident in this reversible research notebook. A skipped
# block additionally skips the MLP, norms and residual attention.
def parameter_count(mod):
    return sum(p.numel() for p in mod.parameters())
full_count=parameter_count(teacher)
structural_prunable=sum(
    parameter_count(original_layers[i]) if mode=="skip_block"
    else parameter_count(original_attention[i]) if mode=="zero" else 0
    for i,(mode,radius) in accepted_plan.items()
)
native_bytes=sum(p.numel()*p.element_size() for p in teacher.parameters())
print("Current research model params:",full_count)
print("Theoretically removable frozen parameters if structurally exported:",
      structural_prunable,"fraction",round(structural_prunable/max(full_count,1),4))
print("Model weights still resident (reversible research):",
      round(native_bytes/2**20,2),"MiB")
# Export plan without copying/loading all 843 MB of pretrained model weights.
optimized_manifest=dict(
    source_checkpoint=SOURCE_MODEL,
    transformer_backend="transformers==4.57.6",
    selected_layer_plan=plan_description(accepted_plan),
    deployment_plan=plan_description(recommended_plan),
    full_attention_layers=FULL_LAYERS,
    sliding_attention_layers=SLIDING_LAYERS,
    attention_logic="native_SDPA_mask_or_skip_layer",
    checkpoint_weights_unmodified=True,
    production_note="Load pretrained checkpoint then apply this manifest with "
                    "the V3.8 notebook's active_plan / wrappers; only after "
                    "quality validation should physical pruning be exported.",
    provisionally_accepted=bool(gates["all_pass"])
)
if EXPORT_ACCEPTED_BLUEPRINT:
    location=OUTPUT_DIR/"v38_optimized_layer_blueprint.json"
    location.write_text(json.dumps(optimized_manifest,indent=2),encoding="utf-8")
    print("Saved reusable layer blueprint:",location)


In [ ]:
#@title 14. Save per-layer ablation report with objective pass/fail and plots
from datetime import datetime,timezone
import matplotlib.pyplot as plt

# Strip heavyweight CPU tensor copies from human-readable file:
# baseline probabilities are only used during the analysis.
report=dict(
    version="3.8-greedy-layerwise-native-sdpa-speed-quality",
    generated_utc=datetime.now(timezone.utc).isoformat(),
    model=SOURCE_MODEL,
    torch_version=torch.__version__,
    gpu=torch.cuda.get_device_name(0),
    seed=SEED,training_updates=0,official_test_used=False,
    train_cases=TRAIN_CASES,dev_cases=DEV_CASES,
    screen_decisions=SCREEN_DECISIONS,search_decisions=SEARCH_DECISIONS,
    audit_decisions=AUDIT_DECISIONS,
    original_layer_kinds=kinds,
    original_local_radius=ORIGINAL_LOCAL_RADIUS,
    screened=screen_results,
    accepted_layerwise_history=greedy_history,
    reverse_refinements=reverse_history,
    selected_probe_blueprint=plan_description(accepted_plan),
    recommended_deployment_blueprint=plan_description(recommended_plan),
    final_probe=final_probe,heldout_audit=final_audit,
    paired_cuda_benchmark=final_timing,
    parameter_count_original=full_count,
    structurally_removable_parameter_upper_bound=structural_prunable,
    pass_fail=gates,
    limitations=[
        "A nonempty blueprint is selected using PROBE only; audit was used only once.",
        "Single T4 run is not proof of speedup: repeat at fixed lengths and seeds.",
        "Skipped parameters remain allocated by the original module while experimenting.",
        "A mask-only change does not guarantee an actual sparse CUDA kernel.",
        "This is zero-shot architecture selection; higher gold accuracy is not guaranteed.",
        "Further distillation and independent official TEST accuracy would be needed for publication.",
        "No weights or shared checkpoint were overwritten.",
    ]
)
saved=OUTPUT_DIR/"v38_layerwise_optimization_ablation_report.json"
saved.write_text(json.dumps(report,indent=2,default=str),encoding="utf-8")
print("Saved comprehensive stepwise report:",saved)
print("Accepted changes",len(accepted_plan),"/",len(ALL_LAYERS))
print("Official TEST used: NO. New training: NO. Final gates:",gates)
# Plot one screened attention-only deletion or best candidate per layer.
screened_cos=[screen_results[str(i)][0]["quality"]["teacher_agreement"]
              for i in ALL_LAYERS]
plt.figure(figsize=(12,4))
plt.plot(ALL_LAYERS,screened_cos,"o-",label="Best isolated 36-item probe option")
plt.axhline(MIN_TEACHER_AGREEMENT,linestyle="--",label="Agreement threshold")
plt.xlabel("Original pretrained layer index")
plt.ylabel("Teacher agreement on 36-item screening sample")
plt.title("V3.8: isolated layer sensitivity before cumulative selection")
plt.ylim(0,1.05)
plt.grid(alpha=.25)
plt.legend()
plt.tight_layout()
plt.show()
